In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# Leitura da Tabela Gold
df_variacao_gold = spark.read.table("workspace.`mvp-engenharia-dados`.combustivel_variacao_gold")
df_bandeira_gold = spark.read.table("workspace.`mvp-engenharia-dados`.combustivel_bandeira_gold")
df_paridade_gold = spark.read.table("workspace.`mvp-engenharia-dados`.combustivel_paridade_gold")


In [0]:

# SIMULAÇÃO 1: RANKING DE EXTREMOS (MENOR E MAIOR PREÇO POR UF E PRODUTO)

# Janelas para classificar o menor e o maior preço médio por UF e Produto
w_menor_gold = Window.partitionBy("estado_sigla", "produto").orderBy(col("preco_medio").asc())
w_maior_gold = Window.partitionBy("estado_sigla", "produto").orderBy(col("preco_medio").desc())

df_min_gold = (df_variacao_gold
    .withColumn("rn", row_number().over(w_menor_gold))
    .filter(col("rn") == 1)
    .select(
        col("estado_sigla"),
        col("produto"),
        col("preco_medio").alias("menor_preco_medio"),
        col("ano_mes").alias("ano_mes_menor_preco")
    )
)

df_max_gold = (df_variacao_gold
    .withColumn("rn", row_number().over(w_maior_gold))
    .filter(col("rn") == 1)
    .select(
        col("estado_sigla"),
        col("produto"),
        col("preco_medio").alias("maior_preco_medio"),
        col("ano_mes").alias("ano_mes_maior_preco")
    )
)

# Consolidação da Consulta
df_simulacao_ranking_1 = (df_min_gold 
    .join(df_max_gold, 
        on=["estado_sigla", "produto"], 
        how="inner")
    .orderBy("produto", "estado_sigla"))

print("\n--- SIMULAÇÃO 1: PREÇO MÉDIO MAIS BARATO E MAIS CARO POR UF (TABELA GOLD) ---\n")
display(df_simulacao_ranking_1)


--- SIMULAÇÃO 1: PREÇO MÉDIO MAIS BARATO E MAIS CARO POR UF (TABELA GOLD) ---



estado_sigla,produto,menor_preco_medio,ano_mes_menor_preco,maior_preco_medio,ano_mes_maior_preco
AC,ETANOL,4.367,2023-01,5.698,2026-03
AL,ETANOL,4.023,2024-01,5.364,2026-04
AM,ETANOL,3.908,2023-01,5.542,2026-03
AP,ETANOL,4.806,2024-06,5.911,2026-05
BA,ETANOL,4.192,2024-01,5.478,2026-04
CE,ETANOL,4.454,2024-01,5.513,2026-04
DF,ETANOL,3.466,2024-01,5.151,2026-03
ES,ETANOL,4.023,2024-01,5.064,2026-04
GO,ETANOL,3.456,2023-12,4.929,2026-01
MA,ETANOL,4.364,2024-02,5.493,2026-04


In [0]:
# SIMULAÇÃO 2: MAIOR VARIAÇÃO PERCENTUAL MENSAL POR REGIÃO

#  Janela para obter a maior variação percentual dentro de cada Região
# (Nota: Assumindo a associação de regiões no catálogo ou via agrupamento)
w_regiao_var = Window.partitionBy("estado_sigla").orderBy(col("variacao_percentual").desc())

df_simulacao_ranking_2 = df_variacao_gold \
    .filter(col("variacao_percentual").isNotNull()) \
    .withColumn("rn", row_number().over(w_regiao_var)) \
    .filter(col("rn") == 1) \
    .select("estado_sigla", "produto", "ano_mes", "preco_medio", "preco_mes_anterior", "variacao_percentual") \
    .orderBy(col("variacao_percentual").desc())

print("\n--- SIMULAÇÃO 2: MAIORES PICOS DE VARIAÇÃO PERCENTUAL (TABELA GOLD) ---\n")
display(df_simulacao_ranking_2)


--- SIMULAÇÃO 2: MAIORES PICOS DE VARIAÇÃO PERCENTUAL (TABELA GOLD) ---



estado_sigla,produto,ano_mes,preco_medio,preco_mes_anterior,variacao_percentual
AM,GASOLINA,2023-03,6.455,5.599,15.29
RN,ETANOL,2026-01,5.352,4.648,15.15
DF,ETANOL,2025-02,4.754,4.203,13.11
MT,ETANOL,2024-04,3.597,3.182,13.04
PI,GASOLINA,2023-03,5.672,5.080,11.65
RO,GASOLINA,2023-03,5.985,5.411,10.61
SE,GASOLINA,2024-02,6.033,5.481,10.07
RS,GASOLINA,2023-03,5.518,5.013,10.07
BA,GASOLINA,2026-03,7.150,6.500,10.00
AP,GASOLINA,2023-03,5.198,4.739,9.69


In [0]:
# SIMULAÇÃO 3: RANKING DE BANDEIRAS MAIS COMPETITIVAS POR REGIÃO

#  Ranquear as bandeiras que ofereceram o menor preço mínimo e menor desvio padrão por região
w_rank_bandeira = Window.partitionBy("regiao_sigla").orderBy(col("preco_minimo").asc(), col("desvio_padrao_preco").asc())

df_simulacao_ranking_3 = df_bandeira_gold \
    .withColumn("rank_competitividade", row_number().over(w_rank_bandeira)) \
    .select(
        "regiao_sigla",
        "rank_competitividade",
        "bandeira",
        "total_pesquisas",
        "preco_minimo",
        "preco_maximo",
        "amplitude_variacao_rs",
        "desvio_padrao_preco"
    ) \
    .filter(col("rank_competitividade") <= 5) \
    .orderBy("regiao_sigla", "rank_competitividade")

print("\n--- SIMULAÇÃO 3: TOP 5 BANDEIRAS MAIS COMPETITIVAS POR REGIÃO (TABELA GOLD) ---\n")
display(df_simulacao_ranking_3)


--- SIMULAÇÃO 3: TOP 5 BANDEIRAS MAIS COMPETITIVAS POR REGIÃO (TABELA GOLD) ---



regiao_sigla,rank_competitividade,bandeira,total_pesquisas,preco_minimo,preco_maximo,amplitude_variacao_rs,desvio_padrao_preco
CO,1,IPIRANGA,24019,2.67,7.29,4.62,1.087
CO,2,BRANCA,46715,2.69,7.49,4.80,1.074
CO,3,RAIZEN,13133,2.69,7.39,4.70,1.1
CO,4,VIBRA,21641,2.75,7.55,4.80,1.106
CO,5,ALESAT,1231,2.79,6.19,3.40,1.019
N,1,BRANCA,23823,3.39,9.29,5.90,0.979
N,2,SABBÃ,8353,3.45,7.59,4.14,0.894
N,3,VIBRA,19588,3.45,8.71,5.26,0.932
N,4,RAIZEN,2399,3.65,8.46,4.81,0.905
N,5,IPIRANGA,16326,3.69,8.99,5.30,0.911


In [0]:
# Classifica os estados a partir da menor paridade (mais vantajoso para o Etanol)
w_paridade = Window.orderBy(col("paridade_percentual").asc())

df_simulacao_paridade = df_paridade_gold \
    .withColumn("rank_vantagem_etanol", row_number().over(w_paridade)) \
    .select(
        col("rank_vantagem_etanol"),
        col("estado_sigla"),
        col("preco_medio_etanol"),
        col("preco_medio_gasolina"),
        col("paridade_percentual"),
        col("recomendacao")
    )

print("\n--- SIMULAÇÃO 4: VIABILIDADE DO ETANOL x GASOLINA (TABELA PARIDADE_GOLD) ---\n")
display(df_simulacao_paridade)


--- SIMULAÇÃO 4: VIABILIDADE DO ETANOL x GASOLINA (TABELA PARIDADE_GOLD) ---



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


rank_vantagem_etanol,estado_sigla,preco_medio_etanol,preco_medio_gasolina,paridade_percentual,recomendacao
1,MT,3.855,6.038,63.84,Compensa ETANOL
2,MS,3.949,5.954,66.33,Compensa ETANOL
3,SP,3.874,5.827,66.49,Compensa ETANOL
4,DF,4.124,6.004,68.69,Compensa ETANOL
5,GO,4.111,5.979,68.77,Compensa ETANOL
6,PR,4.215,6.092,69.19,Compensa ETANOL
7,MG,4.063,5.852,69.43,Compensa ETANOL
8,AM,4.916,6.828,72.00,Compensa GASOLINA
9,AC,5.110,7.088,72.10,Compensa GASOLINA
10,ES,4.451,6.039,73.71,Compensa GASOLINA
